# Practical Solution: Creating a Knowledge Graph in OWLReady2

In [3]:
from owlready2 import *
# owlready2.JAVA_EXE = "C:\\path\\to\\java.exe" #windows users


We begin by importing an ontology. If we are extending an existing ontology then we would import this here. However, here we are creating an ontology from scratch, so we import a blank ontology.

In [4]:
onto = get_ontology("http://www.dummy.info/new.owl")

We will first go through some examples of how to do things manually. Then we will realise that this is very tedious indeed and we will instead read the data in from an external file and create everything dynamically.

We start by creating some *classes* (types) of entity and attach these to the ontology. One way of doing this is use `with` which "opens" the ontology object we have just created and creates our new entities within it. Each entity is created by creating a new class of type `onto.Thing`.

In [5]:

with onto: # This automatically attached the entities to the ontology
    class Staff(Thing):
        pass

    class Student(Thing):
        pass

    class Program(Thing):
        pass

    class Module(Thing):
        pass

    class School(Thing):
        pass

Now we can attach some attributes (properties to each of the entities). For each attribute, we will need to specify which type of entity the property should be attached to, and what the type of the allowable values is:

In [ ]:
with onto:

    class school_name(DataProperty):
        domain = [School]
        range = [str]

    class staff_id(DataProperty):
        domain = [Staff]
        range = [int]

    # Leave this one out initially to demonstrate reparenting
    class staff_title(DataProperty):
        domain = [Staff]
        range = [str]

    class student_id(DataProperty):
        domain = [Student]
        range = [int]

    class person_name(DataProperty):
        domain = [Staff,Student]
        range = [str]

    # Alternative way
    class program_title(Program >> str):
        pass

    class program_id(DataProperty):
        domain = [Module]
        range = [int]

    # Leave this one out initially to demonstrate reparenting
    class program_length(DataProperty):
        domain = [Program]
        range = [str]

    class module_title(DataProperty):
        domain = [Module]
        range = [str]

    class module_id(DataProperty):
        domain = [Module]
        range = [str]

Now we specify some relations. These are specified as `ObjectProperty` and must tell us what type of `Thing` they can be between:

In [7]:
with onto:
    class offers_program(ObjectProperty):
        domain = [School]
        range = [Program]

    class has_module(ObjectProperty):
        domain = [Program]
        range = [Student]

    class is_enrolled_on(ObjectProperty):
        domain = [Student]
        range = [Program]

    class is_taught_by(ObjectProperty):
        domain = [Staff]
        range = [Module]

    class is_directed_by(ObjectProperty):
        domain = [Program]
        range = [Staff]

Let's save the ontology here

In [8]:
onto.save('teaching.rdf')

## Populating the Graph

Now we populate the graph. As we are just exploring, we will only do this sparsely for now to see how it is done. We will then read everything in from files. We'll include:

* One School
* One programme
* One modules
* Two members of academic staff
* One student

Create the entities and assign their attributes

In [ ]:
eeecs = School(name='eeecs', school_name=['EEECS'])
mscaift = Program(name='mscaift', program_title = ['MSc AI Full-time'], program_id = [12345], program_length = ['1 year'])
knowledgeengineering = Module(name='knowledgeengineering', module_title = ["Knowledge Engineering"], module_id = ['ECS8052'])
iainstyles = Staff(name='iainstyles', person_name = ["Iain Styles"], staff_id = [894567], staff_title = ['Professor'])
# a different way to do it
barrydevereux = Staff(name='barrydevereux')
barrydevereux.person_name = ["Barry Devereux"]
barrydevereux.staff_id = [678945]
barrydevereux.staff_title = ['Dr']
alanturing = Student(name='alanturing', person_name = ['Alan Turing'], student_id = [234567])

Now we add in the relations to show how it's done

In [54]:
eeecs.offers_program = [mscaift]
mscaift.has_module = [knowledgeengineering]
alanturing.is_enrolled_on = [mscaift]
knowledgeengineering.is_taught_by = [iainstyles]
mscaift.is_directed_by = [barrydevereux]


In [ ]:
onto.save('teaching.rdf')

## Querying the graph

Now we can construct some simple queries on the graph. 

In [16]:
print(f"{knowledgeengineering.ModuleTitle[0]} is taught by {knowledgeengineering.is_taught_by[0].person_name[0]}")

Knowledge Engineering is taught by Iain Styles


This rapidly becomes inflexible: we want to query classes of object, and this will become very cumbersome. Fortunately there is a mechanism for this. The language designed for this is called SPARQL which is very similar to SQL. Let us see how it works with a few simple examples.

Here is a very simple query that returns everything in the dataset

In [17]:
list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?subject ?predicate ?object
    WHERE{
        ?subject ?predicate ?object
    }
    """))

[[.anonymous, 6, owl.Ontology],
 [www.dummy.info.new.owl, 6, owl.Ontology],
 [new.Staff, 6, 11],
 [new.Staff, 9, owl.Thing],
 [new.Student, 6, 11],
 [new.Student, 9, owl.Thing],
 [new.Program, 6, 11],
 [new.Program, 9, owl.Thing],
 [new.Module, 6, 11],
 [new.Module, 9, owl.Thing],
 [new.School, 6, 11],
 [new.School, 9, owl.Thing],
 [new.school_name, 6, owl.DatatypeProperty],
 [new.school_name, 7, new.School],
 [new.school_name, 8, str],
 [new.staff_id, 6, owl.DatatypeProperty],
 [new.staff_id, 7, new.Staff],
 [new.staff_id, 8, int],
 [new.staff_title, 6, owl.DatatypeProperty],
 [new.staff_title, 7, new.Staff],
 [new.staff_title, 8, str],
 [new.student_id, 6, owl.DatatypeProperty],
 [new.student_id, 7, new.Student],
 [new.student_id, 8, int],
 [new.person_name, 6, owl.DatatypeProperty],
 [new.person_name, 7, new.Staff],
 [new.person_name, 7, new.Student],
 [new.person_name, 8, str],
 [new.ProgramTitle, 6, owl.DatatypeProperty],
 [new.ProgramTitle, 7, new.Program],
 [new.ProgramTitle, 8,


We can refine this query by, for example, restricting the predicate and the object to get specific object for which the predicate with variable object is true.

For example, to get all members of staff, we want to get all objects of type `QUBStaff`:

In [18]:
list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?x
    WHERE{
        ?x rdf:type RDF:Staff
    }
    """))

[[new.iainstyles], [new.barrydevereux]]

Notice that this returns the *object* that satisfies the query.
Now get all students, this time printing the names:

In [ ]:
x = list(default_world.sparql(
    """
    PREFIX RDF: <http://www.dummy.info/new.owl#>
    
    SELECT ?student
    WHERE{
        ?student rdf:type RDF:Student
    }
    """))
print(x[0][0].person_name[0])

Alan Turing


Get all modules and the staff who teach them

In [39]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?module
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff

    }
    """))

[[new.iainstyles, new.knowledgeengineering]]

Our earlier query: all staff taught by an individual

In [ ]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?module
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff
        ?staff ONTO:person_name "Iain Styles"
    }
    """))

[[new.iainstyles, new.knowledgeengineering]]

Get all students taught by each member of staff

In [56]:
list(default_world.sparql(
    """
    PREFIX ONTO: <http://www.dummy.info/new.owl#>
    
    SELECT ?staff ?student
    WHERE{
        ?staff rdf:type ONTO:Staff
        ?student rdf:type ONTO:Student
        ?program rdf:type ONTO:Program
        ?module rdf:type ONTO:Module
        ?module ONTO:is_taught_by ?staff
        ?program ONTO:has_module ?module
        ?student ONTO:is_enrolled_on ?program

    }
    """))

[[new.iainstyles, new.alanturing]]

This is somewhat limiting and we need a bigger set of facts to work with,